# CT-RATE Temporal Labeling v3 — TEXT PRECEDENCE

Silver progression labels for prior→current CT pairs.

**Why v3.** v2 asked the LLM only about findings the structured 18-label table marked
present-in-BOTH. That let the structured presence-diff win unchallenged: a finding the
table called `new`/`resolved` could never be corrected even when the report explicitly
said it increased/decreased. An audit found this fires on **7.4%** of structured labels —
and when the report *does* state a direction, the structured label is wrong **~48%** of
the time (34% for Lymphadenopathy, 16% for Lung nodule).

**v3 fix.** Ask MedGemma about **all 18 findings**, returning `up`/`down`/`same`/
`not_mentioned` with a supporting quote. Then merge with **text winning**:

| Report says | Label | tier |
|---|---|---|
| up / down / same | that direction | `explicit` |
| not_mentioned | fall back to structured presence-diff | `inferred` |

This also separates *explicitly unchanged* from *no comparison found* — v2 lumped both
into `stable`, and only 8.6% of those were genuinely explicit.

**Before running:**
1. Runtime → Change runtime type → **A100 GPU (80GB / High-RAM)**.
2. Accept the license: https://huggingface.co/google/medgemma-27b-text-it
3. Upload **`ctrate_pairs_enriched_v2.csv`**.

In [ ]:
# 1. Deps
!pip -q install -U "transformers>=4.50" accelerate huggingface_hub

In [ ]:
# 2. GPU sanity — expect an A100 with ~80 GB
import torch
assert torch.cuda.is_available(), 'No GPU! Runtime > Change runtime type > A100 GPU'
p = torch.cuda.get_device_properties(0)
print('GPU:', p.name, f'{p.total_memory/1e9:.0f} GB  torch', torch.__version__)
if p.total_memory/1e9 < 70:
    print('WARNING: <70 GB. bf16 27B needs ~54 GB weights + activations; pick the 80 GB A100.')

In [ ]:
# 3. Hugging Face login (gated MedGemma weights)
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# 4. Upload ctrate_pairs_enriched_v2.csv
from google.colab import files
import csv
csv.field_size_limit(10**9)
up = files.upload()
MANIFEST = list(up.keys())[0]
with open(MANIFEST, newline='', encoding='utf-8') as f:
    ROWS = list(csv.DictReader(f))
print(f'{len(ROWS)} pairs loaded from {MANIFEST}')
need = ['presence_changes', 'prior_labels', 'curr_labels']
missing = [c for c in need if c not in ROWS[0]]
assert not missing, f'Missing {missing}. Upload ctrate_pairs_enriched_v2.csv (run scripts/16 first).'
print('v2 manifest OK.')

In [ ]:
# 5. v3 prompt: ask about ALL 18 findings + merge with TEXT PRECEDENCE
import json, re

CANON = ['Medical material','Arterial wall calcification','Cardiomegaly',
  'Pericardial effusion','Coronary artery wall calcification','Hiatal hernia',
  'Lymphadenopathy','Emphysema','Atelectasis','Lung nodule','Lung opacity',
  'Pulmonary fibrotic sequela','Pleural effusion','Mosaic attenuation pattern',
  'Peribronchial thickening','Consolidation','Bronchiectasis','Interlobular septal thickening']

SYSTEM_PROMPT = (
  'You are an expert thoracic radiologist. You are given the PRIOR and CURRENT CT reports '
  'for the same patient. For each finding in a fixed list, report ONLY what the CURRENT '
  'report explicitly says about how it CHANGED relative to the prior study. '
  'Do not infer change from absence of mention. If the current report does not compare '
  'that finding to the prior, say not_mentioned. Respond with ONE JSON object.')

SCHEMA = '''Return ONLY a JSON object:
{"findings": [{"finding": <exact name from the FINDINGS list>, "state": one of ["up","down","same","not_mentioned"], "evidence": <quote from CURRENT report, <=15 words, or "">}]}
Definitions:
- "up"   = the CURRENT report says it is larger / more / increased / progressed / newly appeared.
- "down" = the CURRENT report says it is smaller / fewer / decreased / regressed / resolved / no longer seen.
- "same" = the CURRENT report EXPLICITLY says unchanged / stable / persistent / no significant change.
- "not_mentioned" = the current report does not state any comparison for this finding. USE THIS WHENEVER IN DOUBT.
Rules:
- Include exactly one entry for EVERY finding in the FINDINGS list, in that order.
- "evidence" must be copied verbatim from the CURRENT report; leave "" for not_mentioned.
- Merely describing a finding is NOT a comparison -> not_mentioned.
- Output ONLY the JSON object. No prose, no markdown fences.'''

EXAMPLES = '''Worked example (abbreviated).
PRIOR REPORT: Heart size is increased. No pleural effusion. Subcarinal lymph node 9 mm.
CURRENT REPORT: Heart contour and size are normal. There is bilateral pleural effusion. The subcarinal lymph node short axis is 15 mm, previously 9 mm. Millimetric nodules are noted in both lungs. Emphysematous changes are present.
CORRECT OUTPUT (only the relevant entries shown): {"findings":[{"finding":"Cardiomegaly","state":"down","evidence":"Heart contour and size are normal"},{"finding":"Pleural effusion","state":"up","evidence":"There is bilateral pleural effusion"},{"finding":"Lymphadenopathy","state":"up","evidence":"short axis is 15 mm, previously 9 mm"},{"finding":"Lung nodule","state":"not_mentioned","evidence":""},{"finding":"Emphysema","state":"not_mentioned","evidence":""}]}
Note: nodules and emphysema are DESCRIBED but not COMPARED -> not_mentioned.'''

def curr_text(row):
    return (row.get('curr_findings','') + ' ' + row.get('curr_impression','')).strip()

def build_user(row):
    prior = (row.get('prior_findings','') + ' ' + row.get('prior_impression','')).strip()
    return (EXAMPLES + '\n\nNow do this case.\n'
            f'FINDINGS list (one entry each, this order): {"; ".join(CANON)}\n'
            f'INTERVAL between studies: {row.get("delta_days","")} days\n\n'
            f'PRIOR REPORT:\n{prior or "(none)"}\n\n'
            f'CURRENT REPORT:\n{curr_text(row)}\n\n{SCHEMA}')

def split_sentences(text):
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', (text or '').strip()) if len(s.strip()) > 3]

def extract_json(text):
    text = re.sub(r'```(json)?', '', text)
    i = text.find('{')
    if i < 0: return None
    d = 0
    for j in range(i, len(text)):
        if text[j] == '{': d += 1
        elif text[j] == '}':
            d -= 1
            if d == 0:
                try: return json.loads(text[i:j+1])
                except Exception: return None
    return None

# text state -> 5-class label
STATE_TO_CHANGE = {'up': 'worse', 'down': 'improved', 'same': 'stable'}
# 5-class -> 3-class direction
DIRECTION = {'new':'worsened','worse':'worsened','stable':'stable',
             'improved':'improved','resolved':'improved'}

def combine(row, parsed):
    """TEXT PRECEDENCE: explicit report comparison wins; presence-diff only fills silence."""
    try:
        pres = json.loads(row.get('presence_changes','') or '{}')
    except Exception:
        pres = {}
    states = {}
    if parsed:
        for e in (parsed.get('findings') or []):
            fn, st = e.get('finding'), e.get('state')
            if fn in CANON and st in ('up','down','same','not_mentioned'):
                states[fn] = (st, (e.get('evidence','') or '')[:160])
    out = []
    for f in CANON:
        st, ev = states.get(f, ('not_mentioned', ''))
        p = pres.get(f)  # 'new' | 'resolved' | 'present_both' | None(absent_both)
        if st in ('up','down','same'):
            # ---- TEXT WINS ----
            change = STATE_TO_CHANGE[st]
            # refine using presence only where text is compatible & more specific
            if st == 'up' and p == 'new':
                change = 'new'          # both agree it appeared
            elif st == 'down' and p == 'resolved':
                change = 'resolved'     # both agree it went away
            out.append({'finding': f, 'change': change, 'direction': DIRECTION[change],
                        'tier': 'explicit', 'state': st, 'presence': p or 'absent_both',
                        'agrees_with_presence': bool(
                            (st=='up' and p in ('new','present_both')) or
                            (st=='down' and p in ('resolved','present_both')) or
                            (st=='same' and p == 'present_both')),
                        'evidence': ev})
        else:
            # ---- report silent: fall back to presence-diff, flagged inferred ----
            if p == 'new':
                change = 'new'
            elif p == 'resolved':
                change = 'resolved'
            elif p == 'present_both':
                change = 'no_comparison'   # present both, nothing said -> UNKNOWN (not 'stable')
            else:
                continue                   # absent in both and unmentioned -> skip
            out.append({'finding': f, 'change': change,
                        'direction': DIRECTION.get(change, 'unknown'),
                        'tier': 'inferred', 'state': 'not_mentioned',
                        'presence': p, 'agrees_with_presence': None, 'evidence': ''})
    return out
print('v3 helpers ready — text precedence enabled')

In [ ]:
# 6. Load MedGemma-27B in bf16 straight onto the GPU
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch, time, gc, os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
for _v in ['model', '_o', '_e', 'out', 'enc']:
    if _v in globals():
        try: del globals()[_v]
        except Exception: pass
gc.collect(); torch.cuda.empty_cache()
free_gb = torch.cuda.mem_get_info()[0] / 1e9
print(f'GPU free before load: {free_gb:.1f} GB')
assert free_gb > 60, ('Only %.1f GB free -> a previous model is still resident. '
                      'Runtime > Restart session, then run cells 1-6 again.' % free_gb)
MODEL_ID = 'google/medgemma-27b-text-it'
tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = 'left'
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, attn_implementation='sdpa').to('cuda')
model.config.use_cache = True            # CRITICAL: KV cache, else generation crawls
model.generation_config.use_cache = True
model.eval()
print('model loaded, bf16, use_cache =', model.config.use_cache)

_p = tok.apply_chat_template([{'role':'user','content':'Reply with the single word: ok'}],
                             tokenize=False, add_generation_prompt=True)
_e = tok(_p, return_tensors='pt').to('cuda')
torch.cuda.synchronize(); _t0 = time.time()
with torch.inference_mode():
    _o = model.generate(**_e, max_new_tokens=64, do_sample=False, use_cache=True,
                        pad_token_id=tok.pad_token_id)
torch.cuda.synchronize(); _dt = time.time() - _t0
_ntok = _o.shape[1] - _e['input_ids'].shape[1]
print(f'WARMUP: {_ntok} tokens in {_dt:.1f}s = {_ntok/max(_dt,1e-6):.1f} tok/s')
print('  -> expect >20 tok/s on the A100.')

In [ ]:
# 7. Labeling — writes STRAIGHT TO DRIVE, resumable
from google.colab import drive
import os
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/ct_temporal'
os.makedirs(DRIVE_DIR, exist_ok=True)
OUT = os.path.join(DRIVE_DIR, 'medgemma_labels_v3.jsonl')   # NOTE: v3 filename
LIMIT = 50    # <-- pilot first. Set 0 for all 4,385 once the QC looks right.
BATCH = 8
MAX_NEW = 1400 # 18 findings x ~40 tok each -> needs more room than v2
RESUME = True

rows_all = ROWS if LIMIT == 0 else ROWS[:LIMIT]
done = set()
if RESUME and os.path.exists(OUT):
    for l in open(OUT):
        try:
            d = json.loads(l); done.add((d['prior_volume'], d['curr_volume']))
        except Exception: pass
rows = [r for r in rows_all if (r['prior_volume'], r['curr_volume']) not in done]
print(f'{len(rows_all)} selected, {len(done)} already done, {len(rows)} to label this run')

prompts = [tok.apply_chat_template(
               [{'role':'system','content':SYSTEM_PROMPT},
                {'role':'user','content':build_user(r)}],
               tokenize=False, add_generation_prompt=True)
           for r in rows]

MAXLEN = 8192
tok.truncation_side = 'left'
if prompts:
    lens = sorted(len(tok(p)['input_ids']) for p in prompts)
    print(f'prompt tokens: median={lens[len(lens)//2]} max={lens[-1]} (cap={MAXLEN})')

fout = open(OUT, 'a' if (RESUME and done) else 'w', encoding='utf-8')
n_ok = n_bad = 0
n_batches = (len(prompts) + BATCH - 1) // BATCH
for bi, i in enumerate(range(0, len(prompts), BATCH), 1):
    bp, br = prompts[i:i+BATCH], rows[i:i+BATCH]
    if not bp: break
    enc = tok(bp, return_tensors='pt', padding=True, truncation=True, max_length=MAXLEN).to('cuda')
    print(f'  batch {bi}/{n_batches} generating...', flush=True)
    torch.cuda.synchronize(); t0 = time.time()
    with torch.inference_mode():
        out = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False,
                             use_cache=True, pad_token_id=tok.pad_token_id)
    torch.cuda.synchronize(); dt = time.time() - t0
    texts = tok.batch_decode(out[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
    for row, raw in zip(br, texts):
        parsed = extract_json(raw)
        findings = combine(row, parsed)
        if parsed is not None: n_ok += 1
        else: n_bad += 1
        rec = {'patient': row['patient'], 'prior_volume': row['prior_volume'],
               'curr_volume': row['curr_volume'], 'delta_days': row['delta_days'],
               'findings': findings, 'parse_ok': parsed is not None}
        if parsed is None: rec['raw'] = raw[:2000]
        fout.write(json.dumps(rec) + '\n')
    fout.flush()
    print(f'    {dt:.1f}s  ({i+len(bp)}/{len(prompts)} this run)', flush=True)
fout.close()
print(f'DONE. parse_ok={n_ok} fail={n_bad} -> {OUT} (total lines {sum(1 for _ in open(OUT))})')

In [ ]:
# 8. QC — did text precedence actually change things?
from collections import Counter
recs = [json.loads(l) for l in open(OUT) if l.strip()]
ok = sum(r['parse_ok'] for r in recs)
print(f'records={len(recs)}  parse_ok={ok} ({100*ok/max(len(recs),1):.1f}%)\n')

tier = Counter(); change = Counter(); direction = Counter()
state = Counter(); disagree = Counter()
for r in recs:
    for fd in r['findings']:
        tier[fd['tier']] += 1
        change[fd['change']] += 1
        direction[fd['direction']] += 1
        state[fd['state']] += 1
        if fd['tier'] == 'explicit':
            disagree['agrees' if fd['agrees_with_presence'] else 'DISAGREES with presence'] += 1
tot = sum(tier.values())
print(f'total (pair,finding) labels: {tot:,}  (~{tot/max(len(recs),1):.1f} per pair)')
print('tier          :', dict(tier))
print('5-class change:', dict(change))
print('3-class dir   :', dict(direction))
print('LLM state     :', dict(state))
print('\nAmong EXPLICIT labels, how often did text overrule the structured presence-diff?')
d = sum(disagree.values())
for k, v in disagree.most_common():
    print(f'  {k:<26} {v:>6,} ({100*v/max(d,1):.1f}%)')
print('  -> these are exactly the labels v2 got wrong')

print('\n--- spot-check (first 2 pairs, explicit labels only) ---')
for r in recs[:2]:
    print(f"\n{r['patient']} ({r['delta_days']}d)")
    for fd in r['findings']:
        if fd['tier'] == 'explicit':
            flag = '' if fd['agrees_with_presence'] else '  <-- OVERRODE presence'
            print(f"   {fd['finding']:<30} {fd['change']:<9} [{fd['presence']}]{flag}")
            print(f"      \"{fd['evidence'][:80]}\"")

In [ ]:
# 9. Already on Drive; also pull a local backup
from google.colab import files
import os
print('Drive copy:', OUT, '| exists:', os.path.exists(OUT),
      '| lines:', (sum(1 for _ in open(OUT)) if os.path.exists(OUT) else 0))
files.download(OUT)

## What to check in the pilot QC

1. **`parse_ok` high** (>95%). If low, raise `MAX_NEW` — 18 findings is a long output.
2. **`not_mentioned` should dominate** the LLM `state` counts. Radiologists compare only a
   few findings per study; if the model claims explicit comparisons for most of the 18,
   it's hallucinating comparisons and the prompt needs tightening.
3. **`DISAGREES with presence` > 0** — these are the labels v2 got wrong. The audit
   predicted ~48% of explicit-vs-structured cases disagree.
4. **Spot-check the evidence quotes** — each must be real text from the current report
   that genuinely expresses comparison.

Then set `LIMIT = 0` and run all 4,385. Output persists to Drive and resumes on
disconnect. Send back `medgemma_labels_v3.jsonl` for aggregation.

**New label vocabulary:** `new`, `worse`, `stable`, `improved`, `resolved`, plus
**`no_comparison`** (present in both, report silent = genuinely unknown). Train on it if
you like, but never evaluate on it — v2 wrongly called these `stable`.